# Statistical Rethinking 2026 — Homework A1

## Question A1: The Random Allocation Game (RAG) & The Garden of Forking Data

### Problem Description
To study honesty, behavioral scientists have used an experiment called the **Random Allocation Game (RAG)**. In a RAG, participants are given a single coin. Participants flip the coin, and if the result is heads, they win a small cash prize (like 10 Euros). Participants flip the coin in private—the experimenter cannot see or verify the result, and participants know this.

While it is impossible to know if any individual participant honestly obeys the result of the coin flip, in the aggregate the proportion of prize claims provides information about the frequency of honesty in the sample. For example, if everyone claims the prize, then probably a lot of them are liars.

Suppose **10 participants** play a RAG and **8 of them claim the prize**. Using the **"garden of forking data"** approach:
1. How many ways are there to realize this sample (8 out of 10), if all participants are honest?
2. How many ways, if 5 of the participants are honest?
3. Can you figure out the number of honest participants that maximizes the number of ways to realize the observed sample (8 out of 10)?


## Mathematical Analysis & The Garden of Forking Data

In the "garden of forking data" approach (from Chapter 2 of *Statistical Rethinking*), we count the number of alternative paths (data-generating scenarios) that are consistent with our observed data under different hypotheses.

Here, our observed data is:
* Total participants: $N = 10$
* Claims: $k = 8$
* Non-claims: $N - k = 2$

We define the behaviors of the two types of participants:
1. **Honest Participant**:
   * Flips a coin (2 possible physical outcomes: Heads or Tails).
   * Claims the prize if and only if they flip Heads.
   * *Ways to claim:* $1$ (Heads).
   * *Ways to not claim:* $1$ (Tails).
2. **Dishonest Participant (Liar)**:
   * Flips a coin (2 possible physical outcomes: Heads or Tails), but always claims the prize.
   * *Ways to claim:* $2$ (Heads or Tails).
   * *Ways to not claim:* $0$ (impossible).

Depending on how we define the hypothesis regarding which participants are honest, we can analyze this problem in two ways:

---

### Interpretation 1: Fixed Honesty Identity
We assume a specific, labeled group of $H$ participants are honest, and the remaining $10 - H$ are dishonest. 

Since dishonest participants cannot produce a non-claim, the 2 non-claiming participants **must** be from the honest group.
* We choose which 2 of the $H$ honest participants did not claim: $\binom{H}{2}$ ways.
* These 2 honest participants must have flipped Tails ($1^2 = 1$ way).
* The remaining $H - 2$ honest participants claimed ($1^{H-2} = 1$ way).
* The $10 - H$ dishonest participants claimed ($2^{10-H}$ ways).

Thus, the number of ways to realize the observed sample is:
$$W_{\text{fixed}}(H) = \binom{H}{2} \times 1^2 \times 1^{H-2} \times 2^{10-H} = \binom{H}{2} 2^{10-H}$$

For $H < 2$, $\binom{H}{2} = 0$, meaning it is impossible to observe 2 non-claims if fewer than 2 participants are honest.

---

### Interpretation 2: Variable Honesty Identity
Here, our hypothesis is just the number of honest participants $H$. We do not know who is honest and who is dishonest, so we treat the assignment of honesty as part of the state space.
* Out of the 10 participants, 2 did not claim (and thus must be honest).
* The other 8 participants claimed (they could be honest or dishonest).
* Since exactly $H$ participants are honest in total, and 2 are already identified as the non-claiming honest ones, we must choose $H - 2$ of the other 8 participants to be honest. There are $\binom{8}{H-2}$ ways to designate who is honest.
* For each such designation:
  * The 2 non-claiming honest participants have $1^2 = 1$ way.
  * The $H - 2$ claiming honest participants have $1^{H-2} = 1$ way.
  * The $10 - H$ claiming dishonest participants have $2^{10-H}$ ways.
* We must also account for the $\binom{10}{2} = 45$ ways to choose which 2 of the 10 distinct participants did not claim.

Thus, the total number of ways across all possible designations is:
$$W_{\text{variable}}(H) = \binom{10}{2} \binom{8}{H-2} 2^{10-H} = 45 \binom{8}{H-2} 2^{10-H}$$


In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure styling for publication-quality plots
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'grid.alpha': 0.3,
    'grid.color': '#cccccc',
    'grid.linestyle': '--'
})

def comb(n, k):
    if k < 0 or k > n:
        return 0
    return math.comb(n, k)

# Analytical functions
def ways_fixed(H):
    return comb(H, 2) * (2 ** (10 - H))

def ways_variable(H):
    return comb(10, 2) * comb(8, H - 2) * (2 ** (10 - H))


## Simulation Verification

To ensure our analytical formulas are correct, we can run brute-force simulations of the coin flips for both interpretations.


In [ ]:
# 1. Simulation for Fixed Identity (assuming first H participants are honest)
sim_fixed = []
for H in range(11):
    count = 0
    # Iterate through all 1024 possible coin flip sequences
    for flip_seq in range(1024):
        claims = 0
        for i in range(10):
            coin = (flip_seq >> i) & 1
            if i < H:
                if coin == 1:
                    claims += 1
            else:
                claims += 1  # Dishonest always claims
        if claims == 8:
            count += 1
    sim_fixed.append(count)

# 2. Simulation for Variable Identity (summing over all designations of H honest participants)
import itertools
sim_variable = []
for H in range(11):
    total_count = 0
    for honest_indices in itertools.combinations(range(10), H):
        honest_set = set(honest_indices)
        for flip_seq in range(1024):
            claims = 0
            for i in range(10):
                coin = (flip_seq >> i) & 1
                if i in honest_set:
                    if coin == 1:
                        claims += 1
                else:
                    claims += 1
            if claims == 8:
                total_count += 1
    sim_variable.append(total_count)

# Create a summary DataFrame
df = pd.DataFrame({
    "H (Honest Participants)": list(range(11)),
    "Ways (Fixed Identity - Analytical)": [ways_fixed(h) for h in range(11)],
    "Ways (Fixed Identity - Simulation)": sim_fixed,
    "Ways (Variable Identity - Analytical)": [ways_variable(h) for h in range(11)],
    "Ways (Variable Identity - Simulation)": sim_variable
})

# Display the styled DataFrame
styled_df = df.style.background_gradient(
    cmap="Blues", subset=["Ways (Fixed Identity - Analytical)"]
).background_gradient(
    cmap="Oranges", subset=["Ways (Variable Identity - Analytical)"]
).set_caption(
    "<b>Garden of Forking Data: Ways to Realize 8 Claims out of 10</b>"
).set_properties(**{'text-align': 'center'})

styled_df


## Visualization

Let's plot the number of ways to realize the observed sample (8 out of 10) as a function of the number of honest participants $H$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Custom palette colors
color_fixed = '#3b82f6'    # Modern Indigo/Blue
color_variable = '#f97316' # Warm Orange
accent_color = '#ef4444'   # Coral Red for highlighting max

# Plot 1: Fixed Identity
ax1 = axes[0]
h_vals = np.arange(11)
ways_f = np.array([ways_fixed(h) for h in h_vals])
max_f_mask = (ways_f == ways_f.max())

ax1.plot(h_vals, ways_f, color=color_fixed, linestyle='-', alpha=0.3, zorder=1)
ax1.scatter(h_vals[~max_f_mask], ways_f[~max_f_mask], color=color_fixed, s=80, edgecolors='white', label='Other H', zorder=2)
ax1.scatter(h_vals[max_f_mask], ways_f[max_f_mask], color=accent_color, s=120, edgecolors='white', label='Max Ways', zorder=3)

ax1.set_title("Interpretation 1: Fixed Identity of Honest Group", pad=15)
ax1.set_xlabel("Number of Honest Participants ($H$)", labelpad=10)
ax1.set_ylabel("Number of Ways", labelpad=10)
ax1.set_xticks(h_vals)
ax1.grid(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(frameon=True, facecolor='white', edgecolor='none')

# Annotate peaks
for h, w in zip(h_vals[max_f_mask], ways_f[max_f_mask]):
    ax1.annotate(f"H={h}\n({w} ways)", (h, w), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color=accent_color)

# Plot 2: Variable Identity
ax2 = axes[1]
ways_v = np.array([ways_variable(h) for h in h_vals])
max_v_mask = (ways_v == ways_v.max())

ax2.plot(h_vals, ways_v, color=color_variable, linestyle='-', alpha=0.3, zorder=1)
ax2.scatter(h_vals[~max_v_mask], ways_v[~max_v_mask], color=color_variable, s=80, edgecolors='white', label='Other H', zorder=2)
ax2.scatter(h_vals[max_v_mask], ways_v[max_v_mask], color=accent_color, s=120, edgecolors='white', label='Max Ways', zorder=3)

ax2.set_title("Interpretation 2: Variable Identity of Honest Group", pad=15)
ax2.set_xlabel("Number of Honest Participants ($H$)", labelpad=10)
ax2.set_ylabel("Number of Ways", labelpad=10)
ax2.set_xticks(h_vals)
ax2.grid(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(frameon=True, facecolor='white', edgecolor='none')

# Annotate peaks
for h, w in zip(h_vals[max_v_mask], ways_v[max_v_mask]):
    ax2.annotate(f"H={h}\n({w:,} ways)", (h, w), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color=accent_color)

plt.tight_layout()
plt.savefig('homework_a1_rag_plot.png', dpi=300, bbox_inches='tight')
plt.show()


## Summary of Answers

### 1. How many ways are there if all participants are honest ($H = 10$)?
* **Interpretation 1 (Fixed Identity):** There are **$45$ ways** to realize the observed sample.
* **Interpretation 2 (Variable Identity):** There are **$45$ ways** to realize the observed sample.
* *Note:* Since everyone is honest, there is only $1$ designation of who is honest (all 10). Both models converge to $\binom{10}{2} = 45$ ways of choosing which 2 participants flipped Tails.

### 2. How many ways are there if 5 of the participants are honest ($H = 5$)?
* **Interpretation 1 (Fixed Identity):** There are **$320$ ways** to realize the observed sample.
* **Interpretation 2 (Variable Identity):** There are **$80,640$ ways** to realize the observed sample.
* *Note:* In Interpretation 2, we sum over all $\binom{8}{3} = 56$ possible designations of who is honest among the 8 claiming participants, leading to $56 \times 320 = 80,640$ ways.

### 3. Which number of honest participants $H$ maximizes the number of ways?
* **Interpretation 1 (Fixed Identity):** The maximum occurs at **$H = 3$** and **$H = 4$**, both yielding **$384$ ways**.
* **Interpretation 2 (Variable Identity):** The maximum occurs at **$H = 4$** and **$H = 5$**, both yielding **$80,640$ ways**.
* *Bayesian Context:* In a Bayesian setting with a flat prior, the number of ways is directly proportional to the likelihood of the data given the parameter $H$. Therefore, $H=3$ or $H=4$ (Interpretation 1) or $H=4$ or $H=5$ (Interpretation 2) represents the maximum likelihood estimate (MLE) or mode of the posterior distribution for the number of honest participants in the sample.
